

# Data Splitting Strategy
# To ensure proper model training and evaluation, the dataset is divided into three subsets:

# Training Set (70%)
# Used to train the model and learn underlying patterns in the data.

# Validation Set (15%)
# Used for tuning hyperparameters and preventing overfitting during training.

# Test Set (15%)
# Used for final evaluation to assess the model’s performance on unseen data.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, RobustScaler,LabelEncoder
import joblib
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/cleaned_1.csv")

X = df.drop(columns=["diabetes"])
y = df["diabetes"]

FileNotFoundError: [Errno 2] No such file or directory: '../../data/cleaned_1.csv'

Split the data into 70% Train and 30% temp
we use statify = y (Target Feature) because there is Data Imbalance so it is important to keep the same data ratio in train ,validation and test

we use random_state to ensure consistent results across multiple runs

In [3]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state = 42
)

NameError: name 'X' is not defined

In [ ]:
# Split the remaining 30% equally into Validation and Test sets
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)
print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

In [ ]:
y_train.head(10)

In [ ]:
# Save Into Data Folder
train_df = X_train.copy()
train_df["diabetes"] = y_train

val_df = X_val.copy()
val_df["diabetes"] = y_val

test_df = X_test.copy()
test_df["diabetes"] = y_test


train_df.to_csv("../../data/split/train.csv", index=False)
val_df.to_csv("../../data/split/validation.csv", index=False)
test_df.to_csv("../../data/split/test.csv", index=False)

Data Transformation

In [ ]:
train_df = pd.read_csv("../../data/split/train.csv")
val_df = pd.read_csv("../../data/split/validation.csv")
test_df = pd.read_csv("../../data/split/test.csv")
x_train = train_df.drop("diabetes", axis=1)
y_train = train_df["diabetes"]

x_val = val_df.drop("diabetes", axis=1)
y_val = val_df["diabetes"]

x_test = test_df.drop("diabetes", axis=1)
y_test = test_df["diabetes"]

In [ ]:
def discretize(df):
    df["hba1c_category"] = pd.cut(
            df['HbA1c_level'],
            bins=[0, 5.7, 6.5, float("inf")],
            labels=["Normal", "Prediabetes", "Diabetes"],
            right= False
        )
    
    df['bmi_category'] = pd.cut(
            df['bmi'],
            bins=[0, 18.5, 25, 30, float("inf")],
            labels=["Underweight", "Healthy Weight", "Overweight", "Obesity"],
            right= False
        )
    
    df['blood_glucose_category'] = pd.cut(
            df['blood_glucose_level'],
            bins=[70, 140, 200, float("inf")],
            labels=["Normal", "Prediabetes", "Possible Diabetes"],
            right= False
        )
    
    return df

Feature Encoding
Gender Binary Encoding
Converts categorical gender to numeric binary
Female = 0 | Male = 1

In [ ]:
x_train["gender"] = x_train["gender"].replace({"Female": 0, "Male": 1})
x_val["gender"] = x_val["gender"].replace({"Female": 0, "Male": 1})
x_test["gender"] = x_test["gender"].replace({"Female": 0, "Male": 1})

Ordinal Encoding for Discretized Categories
Maps clinically ordered string categories to integers
Preserves natural risk ordering: lowest risk = 0 → highest risk = n

In [ ]:
def ordinal_encode_categories(df):

    hba1c_mapping = {
        "Normal"      : 0,
        "Prediabetes" : 1,
        "Diabetes"    : 2
    }

    bmi_mapping = {
        "Underweight"    : 0,
        "Healthy Weight" : 1,
        "Overweight"     : 2,
        "Obesity"        : 3
    }

    blood_glucose_mapping = {
        "Normal"            : 0,
        "Prediabetes"       : 1,
        "Possible Diabetes" : 2
    }

    df["hba1c_category"]         = df["hba1c_category"].map(hba1c_mapping).astype(int)
    df["bmi_category"]           = df["bmi_category"].map(bmi_mapping).astype(int)
    df["blood_glucose_category"] = df["blood_glucose_category"].map(blood_glucose_mapping).astype(int)

    return df

In [ ]:
# Feature Engineering
# Arithmetic Interaction Features

def add_engineered_features(df):
    # arithimetic features     
    df["glucose_hba1c_interaction"] = df["blood_glucose_level"] * df["HbA1c_level"]
    df["age_hba1c_interaction"] = df["age"] * df["HbA1c_level"]
    df["age_bmi_interaction"] = df["age"] * df["bmi"]
    df["bmi_hba1c_interaction"] = df["bmi"] * df["HbA1c_level"]
    df["age_glucose_interaction"] = df["age"] * df["blood_glucose_level"]
    
    # Boolean and Logical Combinations
    df["high_hba1c_flag"] = (df["HbA1c_level"] >= 6.6).astype(int)
    df["senior_flag"] = (df["age"] >= 60).astype(int)
    df["cardio_risk_flag"] = ((df["hypertension"] == 1) | (df["heart_disease"] == 1)).astype(int)

    return df

Binary Flags & Composite Risk Features
Converts clinical thresholds and domain knowledge into binary signals (0/1), making categorical risk boundaries explicitly available to the model.

In [ ]:
def add_engineered_features(df):
    # arithimetic features     
    df["glucose_hba1c_interaction"] = df["blood_glucose_level"] * df["HbA1c_level"]
    df["age_hba1c_interaction"] = df["age"] * df["HbA1c_level"]
    df["age_bmi_interaction"] = df["age"] * df["bmi"]
    df["bmi_hba1c_interaction"] = df["bmi"] * df["HbA1c_level"]
    df["age_glucose_interaction"] = df["age"] * df["blood_glucose_level"]
    
    # Boolean and Logical Combinations
    df["high_hba1c_flag"] = (df["HbA1c_level"] >= 6.6).astype(int)
    df["senior_flag"] = (df["age"] >= 60).astype(int)
    df["cardio_risk_flag"] = ((df["hypertension"] == 1) | (df["heart_disease"] == 1)).astype(int)

    return df

In [ ]:
# Data Normalization
preprocessor = ColumnTransformer(
    transformers=[
        ("smoking_ohe", OneHotEncoder(handle_unknown="ignore",drop="first"), ["smoking_history"]),
        ("age_minmax", MinMaxScaler(), ["age"]),
        ("robust_features", RobustScaler(), ["bmi",
                                              "HbA1c_level",
                                              "blood_glucose_level",
                                              "glucose_hba1c_interaction",
                                              "age_hba1c_interaction",
                                              "age_bmi_interaction",
                                              "bmi_hba1c_interaction",
                                              "age_glucose_interaction"])
    ],
    remainder="passthrough", # to keep the rest of the columns unchanged
    verbose_feature_names_out=False  # to prevent adding the transformer name as a prefix to the feature names
    # smoking_history_No Info insted of smoking_ohe__smoking_history_No Info
)
joblib.dump(preprocessor, "preprocessor.pkl")

In [ ]:
# Transform Training data
x_train_featured = add_engineered_features(x_train)
x_train_processed = preprocessor.fit_transform(x_train_featured) # returns numpy array not Pandas DataFrame so we need to convert it back to DataFrame
joblib.dump(preprocessor, "preprocessor.pkl")
feature_names = preprocessor.get_feature_names_out()
x_train_processed = pd.DataFrame(
    x_train_processed,
    columns=feature_names,
    index=x_train.index
)
x_train_processed.head()

In [ ]:
# Transform Validation data
x_val = add_engineered_features(x_val)
preprocessor = joblib.load("preprocessor.pkl")
x_val_processed = preprocessor.transform(x_val)
x_val_processed = pd.DataFrame(
    x_val_processed,
    columns=feature_names,   # same feature names from train
    index=x_val.index
)

x_val_processed.head()

In [ ]:
# Transform Cleaned Dataset For EDA stage
cleaned = pd.read_csv("../../data/cleaned/cleaned_1.csv")
cleaned_and_binned = discretize(cleaned)
cleaned_binned_engineered = add_engineered_features(cleaned_and_binned)

cleaned_binned_engineered.to_csv("../../data/EDA/transformed_for_eda.csv", index=False)
train_processed_df = x_train_processed.copy()
train_processed_df["diabetes"] = y_train.values

val_processed_df = x_val_processed.copy()
val_processed_df["diabetes"] = y_val.values

train_processed_df.to_csv("../../data/preprocessed/train_preprocessed.csv", index=False)
val_processed_df.to_csv("../../data/preprocessed/validation_preprocessed.csv", index=False)

train_processed_df.shape